In [1]:
import pandas as pd
import numpy as np
import datetime as dt
import glob

from XMA_finder import XMA_finder
from Cluster_CDF_conv import Cluster_cdf_conv
#from histo_plot_lower_vmax import histo_plot
from matplotlib import pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates

#import modules
from matplotlib import pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib import cm
import matplotlib.colors as colors
from merka05_surface_eq_array_GIPM import merka05_surface_eq_array_GIPM
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator)
import matplotlib.ticker as ticker

In [3]:
#import Cluster integrated CSVs, separated by spacecraft in order to match w/Fourier spectra
#import C1/C2/C3/C4 csvs and determine min/max power
#then go back to update the colourmap norm choice

##load Cluster CSVs

c1_file_list = []
c2_file_list = []
c3_file_list = []
c4_file_list = []

path = "/Users/roseatkinson/Documents/Cluster_Integrated_CSVs/Integrated_CSVs_2001/**"

for path in glob.glob(path, recursive=True):
    if '.csv' in path:
        if 'C1' in path:
            c1_file_list.append(path)
        if 'C2' in path:
            c2_file_list.append(path)   
        if 'C3' in path:
            c3_file_list.append(path)
        if 'C4' in path:
            c4_file_list.append(path)

cl_csvs = [c1_file_list, c2_file_list, c3_file_list, c4_file_list]
            
c1_dfs = []
c2_dfs = []
c3_dfs = []
c4_dfs = []

cl_dfs = [c1_dfs, c2_dfs, c3_dfs, c4_dfs]

for csv_list, df_list in zip(cl_csvs, cl_dfs):
    for file in csv_list:
        df = pd.read_csv(file,encoding='utf-8')
        df['datetime'] = pd.to_datetime(df['datetime'], format='mixed')
        df.set_index('datetime', inplace = True)
        df_list.append(df)

cl_1_df = pd.concat(c1_dfs)
cl_2_df = pd.concat(c2_dfs)
cl_3_df = pd.concat(c3_dfs)
cl_4_df = pd.concat(c4_dfs)

#Filtering: remove OMNI too far, current sheets, & null GIPM entries
cl_1_filtered = cl_1_df.loc[(cl_1_df['OMNI Dist from X line (mean)'] < 70) & (cl_1_df['Max IMF Deviation'] < 60) & (cl_1_df['GIPM X (OMNI mean)'].notnull())]
cl_2_filtered = cl_2_df.loc[(cl_2_df['OMNI Dist from X line (mean)'] < 70) & (cl_2_df['Max IMF Deviation'] < 60) & (cl_2_df['GIPM X (OMNI mean)'].notnull())]
cl_3_filtered = cl_3_df.loc[(cl_3_df['OMNI Dist from X line (mean)'] < 70) & (cl_3_df['Max IMF Deviation'] < 60) & (cl_3_df['GIPM X (OMNI mean)'].notnull())]
cl_4_filtered = cl_4_df.loc[(cl_4_df['OMNI Dist from X line (mean)'] < 70) & (cl_4_df['Max IMF Deviation'] < 60) & (cl_4_df['GIPM X (OMNI mean)'].notnull())]


In [10]:
#first off: looking at polarisation by taking ratio of power in two transverse components 
#will need to match the spectra to the Cluster GIPM measurement

#/Users/roseatkinson/Documents/Spectra_2001
#example file:
#'/Users/roseatkinson/Documents/Spectra_2001/FS_2001-02-02 19:14:00_C4.csv'
#iterate through filtered cluster datasets
#make new dataframe with the datetimes as the index and the results as columns
#then later join on df index to original df with the rest of the cluster data
cl_1_datetimes = []
cl_1_peak_freqs = []
cl_1_ellipticity = []

for datetime in cl_1_filtered.index[0:5]:
    filename = '/Users/roseatkinson/Documents/Spectra_2001/FS_'+ str(datetime) + '_C1.csv'
    f_df = pd.read_csv(filename)
    spectra_dfs.append(f_df)
    #find the peak power for compressive frequency
    
    
    ## Compute the power into the required frequency range using midpoint integration
    F_LOW,F_HIGH = 7e-3,1e-1
    #make np arrays of columns first
    freq = f_df['Freq'].to_numpy()
    p_perp_1 = f_df['Perp 1 Power'].to_numpy()
    p_perp_2 = f_df['Perp 2 Power'].to_numpy()
    mask = (freq>F_LOW) & (freq<F_HIGH)
    delta_f = freq[1] - freq[0]
    P_perp_1 = np.sum(p_perp_1[mask])*delta_f
    P_perp_2 = np.sum(p_perp_2[mask])*delta_f

    if P_perp_1 > P_perp_2:
        ratio = P_perp_1/P_perp_2
        ratio_perp_power.append(ratio)
    else:
        ratio = P_perp_2/P_perp_1
        ratio_perp_power.append(ratio)
        
    

In [ ]:
#second: plotting maximum frequency in the ULF band

In [11]:
ratio_perp_power

[np.float64(4.2306909268571244),
 np.float64(2.0844010661848413),
 np.float64(1.338181486355005),
 np.float64(4.012583254317602),
 np.float64(3.0361356670799506)]